# Track B: The Engineering — Noise, Transpilation, and Cost

**Plan C — Parallel Tracks**

This track is about making quantum circuits work on noisy hardware. You will learn how noise degrades the encoded state, how transpilation maps logical circuits to physical hardware, and how the scoring formula balances quality against cost.

> **Dashboard:** Open `00_dashboard.ipynb` to interactively compare settings.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from math import pi, sqrt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, DensityMatrix, state_fidelity
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

from autoresearch_quantum.codes.four_two_two import (
    build_preparation_circuit, encoded_magic_statevector,
    STABILIZERS, MEASUREMENT_OPERATORS, DATA_QUBITS,
)
from autoresearch_quantum.experiments.encoded_magic_state import build_circuit_bundle
from autoresearch_quantum.models import (
    ExperimentSpec, RungConfig, EvaluationMetrics,
    QualityWeights, CostWeights, ScoreConfig, SearchSpaceConfig,
    TierPolicyConfig, HardwareConfig,
)
from autoresearch_quantum.execution.local import LocalCheapExecutor
from autoresearch_quantum.execution.analysis import (
    logical_magic_witness, stability_score, summarize_context,
    local_memory_records,
)
from autoresearch_quantum.execution.backends import resolve_backend, backend_metadata
from autoresearch_quantum.execution.transpile import (
    transpile_circuits, count_two_qubit_gates, circuit_metadata, runtime_estimate,
)
from autoresearch_quantum.scoring.score import (
    score_metrics, weighted_acceptance_cost, factory_throughput_score,
)
from autoresearch_quantum.config import load_rung_config

print("All imports successful.")

In [ ]:
from autoresearch_quantum.teaching import LearningTracker
from autoresearch_quantum.teaching.assess import quiz, predict_choice, reflect, order, checkpoint_summary
tracker = LearningTracker("plan_c_track_b")
print("Learning tracker active.")

---
## 1. Ideal vs Noisy Simulation

Let us build an encoded magic state circuit and run it in two regimes: perfect (no noise) and realistic (fake_brisbane noise model).

In [ ]:
# Build the circuit
spec = ExperimentSpec(
    rung=1, seed_style="h_p", encoder_style="cx_chain",
    verification="both", postselection="all_measured",
    target_backend="fake_brisbane", noise_backend="fake_brisbane",
    optimization_level=2, shots=512, repeats=1,
)
bundle = build_circuit_bundle(spec)
backend = resolve_backend("fake_brisbane")

# Transpile
transpiled = transpile_circuits([bundle.acceptance], spec, backend)[0]

# Ideal
ideal_sim = AerSimulator()
ideal_result = ideal_sim.run(transpiled, shots=512, memory=True, seed_simulator=42).result()

# Noisy
noise_model = NoiseModel.from_backend(backend)
noisy_sim = AerSimulator(
    noise_model=noise_model,
    basis_gates=noise_model.basis_gates,
    coupling_map=backend.coupling_map,
)
noisy_result = noisy_sim.run(transpiled, shots=512, memory=True, seed_simulator=42).result()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
plot_histogram(ideal_result.get_counts(), ax=ax1, title="Ideal")
plot_histogram(noisy_result.get_counts(), ax=ax2, title="Noisy (fake_brisbane)")
plt.tight_layout()
plt.show()

In [ ]:
predict_choice(tracker, "q1_noise_histogram",
    question="Comparing the ideal and noisy histograms: what is the main visible difference?",
    options=[
        "The noisy histogram has fewer bars",
        "The noisy histogram spreads probability across many more bitstrings",
        "They look identical",
    ],
    correct=1, section="1. Ideal vs noisy", bloom="understand",
    explanation="Noise causes probability to leak from the valid codewords to other basis states. This spreading is the visual signature of decoherence.")

> **Observe:** The ideal histogram concentrates on a few bitstrings (valid codewords). Noise spreads probability across many bitstrings — errors kick the state out of the codespace.

---
## 2. What Is a Noise Model?

A noise model describes the errors each gate introduces. The fake_brisbane backend models the real IBM Brisbane device.

In [ ]:
meta = backend_metadata(backend)
print(f"Backend: {meta['name']}")
print(f"Qubits:  {meta['num_qubits']}")
print(f"Native gates: {meta['operation_names'][:8]}...")
print(f"Coupling edges: {meta['coupling_edges']}")

noise_dict = noise_model.to_dict()
print(f"\nNoise model: {len(noise_dict['errors'])} error channels")

# Show a sample of error rates
error_types = {}
for err in noise_dict['errors']:
    etype = err['type']
    error_types[etype] = error_types.get(etype, 0) + 1
print("\nError types:")
for etype, count in error_types.items():
    print(f"  {etype}: {count} channels")

In [ ]:
quiz(tracker, "q2_native_gates",
    question="The hardware has native gates like CX, SX, RZ. What happens to non-native gates like H?",
    options=[
        "They are executed directly",
        "The transpiler decomposes them into native gate sequences",
        "They cause an error",
    ],
    correct=1, section="2. Backend", bloom="understand",
    explanation="The transpiler converts all gates to the hardware's native set. H becomes SX + RZ. This decomposition adds gates and thus noise.")
checkpoint_summary(tracker, "2. Backend")

---
## 3. Transpilation: Logical to Physical

The logical circuit uses abstract gates (H, CX, P). The physical hardware only supports specific native gates. **Transpilation** maps one to the other, adding SWAP gates for connectivity and decomposing gates into the native set.

Qiskit offers optimization levels 1-3, trading transpile time for circuit quality.

In [ ]:
# Transpile at different optimization levels
prep = build_preparation_circuit("h_p", "cx_chain")

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
stats = {}
for i, opt in enumerate([1, 2, 3]):
    opt_spec = spec.with_updates(optimization_level=opt)
    t_circ = transpile_circuits([prep], opt_spec, backend)[0]
    twoq = count_two_qubit_gates(t_circ)
    d = t_circ.depth()
    stats[opt] = {"2q_gates": twoq, "depth": d, "size": t_circ.size()}
    print(f"opt_level={opt}: 2Q gates={twoq}, depth={d}, total gates={t_circ.size()}")
    t_circ.draw("mpl", ax=axes[i], style="iqp")
    axes[i].set_title(f"Optimization Level {opt}")

plt.tight_layout()
plt.show()

In [ ]:
predict_choice(tracker, "q3_opt_levels",
    question="Higher transpilation optimization levels reduce gate count. Is this always better?",
    options=[
        "Yes \u2014 fewer gates always means less noise",
        "Not necessarily \u2014 aggressive optimization may reroute qubits in ways that increase cross-talk",
        "No \u2014 lower optimization is always more reliable",
    ],
    correct=1, section="3. Transpilation", bloom="analyze",
    explanation="Optimization involves trade-offs. Reducing gates helps, but qubit routing decisions can place operations on noisier connections.")

### Numerical Exercise: Cost Calculation

Using the transpilation stats printed above, compute the two-qubit gate contribution to cost.

The cost weight for two-qubit gates is \\(w_{2q} = 0.08\\). If opt_level=1 has, say, \\(n\\) two-qubit gates, the contribution is \\(0.08 \\times n\\).

> **Key Insight:** Higher optimization levels generally reduce gate count and depth, but the effect is non-monotonic — it depends on the specific circuit and backend topology. Each two-qubit gate is ~10x noisier than a single-qubit gate, so fewer = better.

---
## 4. The Cost Model

The scoring formula penalizes circuit cost:

$$\text{cost} = \text{base} + w_{2q} \cdot n_{2q} + w_d \cdot d + w_s \cdot s + w_r \cdot r + w_q \cdot q$$

| Symbol | Meaning | Typical weight |
|---|---|---|
| $n_{2q}$ | Two-qubit gate count | 0.08 |
| $d$ | Circuit depth | 0.01 |
| $s$ | Total shot count | 0.0002 |
| $r$ | Runtime estimate | 0.015 |
| $q$ | Queue cost proxy | 0.30 |

In [ ]:
quiz(tracker, "q4_cost_driver",
    question="What is the dominant cost driver for most quantum circuits?",
    options=[
        "Single-qubit gate count",
        "Two-qubit (CX/CZ) gate count \u2014 these are 10-100x noisier than single-qubit gates",
        "Classical post-processing time",
    ],
    correct=1, section="4. Cost model", bloom="apply",
    explanation="Two-qubit gates have error rates 10-100x higher than single-qubit gates on current hardware. Minimizing 2Q count is the primary optimization target.")
checkpoint_summary(tracker, "4. Cost model")

In [ ]:
# Compute cost for each optimization level
rung_config = load_rung_config("../../configs/rungs/rung1.yaml")
cw = rung_config.score.cost_weights

for opt, s in stats.items():
    cost = (rung_config.score.base_cost
            + cw.two_qubit_count * s["2q_gates"]
            + cw.depth * s["depth"]
            + cw.shot_count * 512
            + cw.runtime_estimate * (s["depth"] + 3 * s["2q_gates"]))
    print(f"opt_level={opt}: cost = {cost:.3f}  (2q contrib = {cw.two_qubit_count * s['2q_gates']:.3f}, depth contrib = {cw.depth * s['depth']:.3f})")

---
## 5. Acceptance Rate Under Noise

Postselection keeps only shots where stabilizer syndrome = 0. Under noise, many shots fail.

In [ ]:
quiz(tracker, "q5_acceptance_meaning",
    question="Acceptance rate of 60% means:",
    options=[
        "60% of the circuit gates succeeded",
        "60% of shots passed the stabilizer check \u2014 40% had detectable errors",
        "The state has 60% fidelity",
    ],
    correct=1, section="5. Acceptance", bloom="apply",
    explanation="40% of shots triggered a syndrome flag and were discarded. You need ~1.7x the shots to get the same number of clean data points.")
checkpoint_summary(tracker, "5. Acceptance")

In [ ]:
# Run full evaluation at different noise levels (via optimization levels as proxy)
rung_config = load_rung_config("../../configs/rungs/rung1.yaml")
executor = LocalCheapExecutor()

opt_results = {}
for opt in [1, 2, 3]:
    s = ExperimentSpec(
        rung=1, seed_style="h_p", encoder_style="cx_chain",
        verification="both", postselection="all_measured",
        target_backend="fake_brisbane", noise_backend="fake_brisbane",
        optimization_level=opt, shots=256, repeats=1,
    )
    result = executor.evaluate(s, rung_config)
    opt_results[opt] = result
    m = result.metrics
    print(f"opt={opt}: acceptance={m.acceptance_rate:.3f}  witness={m.logical_magic_witness:.3f}  "
          f"fidelity_noisy={m.noisy_encoded_fidelity:.3f}  score={result.score:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

labels = [f"Level {o}" for o in [1, 2, 3]]

# Acceptance rate
axes[0].bar(labels, [opt_results[o].metrics.acceptance_rate for o in [1, 2, 3]],
            color=["#e74c3c", "#2ecc71", "#3498db"])
axes[0].set_title("Acceptance Rate")
axes[0].set_ylim(0, 1)

# Quality metrics
x = np.arange(3)
w = 0.25
axes[1].bar(x-w, [opt_results[o].metrics.logical_magic_witness for o in [1,2,3]], w, label="Witness")
axes[1].bar(x, [opt_results[o].metrics.noisy_encoded_fidelity for o in [1,2,3]], w, label="Noisy Fid.")
axes[1].bar(x+w, [opt_results[o].metrics.acceptance_rate for o in [1,2,3]], w, label="Accept.")
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels)
axes[1].set_title("Quality Metrics")
axes[1].legend(fontsize=8)

# Score
axes[2].bar(labels, [opt_results[o].score for o in [1, 2, 3]],
            color=["#e74c3c", "#2ecc71", "#3498db"])
axes[2].set_title("Final Score")

plt.tight_layout()
plt.show()

> **Key Insight:** The "best" optimization level is not always the highest — it depends on how the backend's noise profile interacts with the transpiled circuit layout.

---
## 6. Noisy Fidelity via Density Matrix

State fidelity measures overlap between the noisy output and the ideal encoded T-state.

In [ ]:
# Noisy fidelity via density matrix on the LOGICAL (untranspiled) 4-qubit circuit.
# The transpiled circuit uses 127 qubits (full backend), making density matrix
# simulation infeasible (2^127 x 2^127 matrix). Instead we use the 4-qubit circuit
# with a simplified noise model.

target = encoded_magic_statevector()
prep = build_preparation_circuit("h_p", "cx_chain")

from qiskit_aer.noise import NoiseModel as NM2, depolarizing_error
simple_noise = NM2()
simple_noise.add_all_qubit_quantum_error(depolarizing_error(0.001, 1), ['h', 'p', 'rz', 'ry', 'sx', 'x', 'u'])
simple_noise.add_all_qubit_quantum_error(depolarizing_error(0.01, 2), ['cx'])

dm_circuit = prep.copy()
dm_circuit.save_density_matrix()
density_sim = AerSimulator(method="density_matrix", noise_model=simple_noise)
dm_result = density_sim.run(dm_circuit).result()
noisy_dm = dm_result.data(0)["density_matrix"]

ideal_fid = state_fidelity(Statevector.from_instruction(prep), target)
noisy_fid = state_fidelity(noisy_dm, target)

print(f"Ideal fidelity:  {ideal_fid:.6f}")
print(f"Noisy fidelity:  {noisy_fid:.6f}")
print(f"Fidelity loss:   {ideal_fid - noisy_fid:.6f}")

---
## 7. Failure Mode Classification

The harness classifies each experiment by its **dominant failure mode**:

| Mode | Trigger | Meaning |
|---|---|---|
| postselection collapse | acceptance < 0.45 | Too many errors detected |
| logical witness erosion | witness < 0.65 | Magic property severely degraded |
| noise sensitivity | stability < 0.75 | Results vary wildly between repeats |
| transpile cost explosion | 2q > 60 or depth > 120 | Circuit too expensive |

In [ ]:
order(tracker, "q6_failure_severity",
    instruction="Rank failure modes from least to most severe:",
    items=["high_cost", "poor_acceptance", "low_magic_witness"],
    correct_order=["high_cost", "poor_acceptance", "low_magic_witness"],
    section="7. Failure modes", bloom="analyze",
    explanation="High cost is fixable (optimize gates). Poor acceptance wastes shots. Low witness means the T-state character is lost \u2014 the experiment's purpose has failed.")
checkpoint_summary(tracker, "7. Failure modes")

In [ ]:
for opt in [1, 2, 3]:
    m = opt_results[opt].metrics
    print(f"opt={opt}: {m.dominant_failure_mode:30s}  "
          f"(accept={m.acceptance_rate:.2f}, witness={m.logical_magic_witness:.2f}, "
          f"stability={m.stability_score:.2f}, 2q={m.two_qubit_count}, depth={m.depth})")

---
## 8. The Full Scoring Formula

$$\text{score} = \frac{\text{quality} \times \text{acceptance\_rate}}{\text{cost}}$$

Quality is a weighted average of metrics. The rung1 config weights noisy fidelity highest (0.40), followed by logical witness (0.25).

In [ ]:
reflect(tracker, "q7_score_manual",
    question="You see the score computed manually from quality, acceptance, and cost. Which component dominates and why?",
    section="8. Scoring", bloom="evaluate",
    model_answer="It depends on the noise regime. At low noise: cost dominates (quality and acceptance are both near 1). At high noise: acceptance dominates (many shots rejected). The score formula surfaces whichever factor is the bottleneck.")

In [ ]:
# Manual computation for the best optimization level
best_opt = max(opt_results, key=lambda o: opt_results[o].score)
m = opt_results[best_opt].metrics
sc = rung_config.score
w = sc.cheap_quality

components = [
    ("ideal_fidelity", w.ideal_fidelity, m.ideal_encoded_fidelity),
    ("noisy_fidelity", w.noisy_fidelity, m.noisy_encoded_fidelity),
    ("logical_witness", w.logical_witness, m.logical_magic_witness),
    ("codespace_rate", w.codespace_rate, m.codespace_rate),
    ("stability_score", w.stability_score, m.stability_score),
    ("spectator_alignment", w.spectator_alignment,
     (1 + m.spectator_logical_z) / 2 if m.spectator_logical_z is not None else None),
]

print(f"Quality components (opt_level={best_opt}):")
print(f"{'Component':25s} {'Weight':>8s} {'Value':>8s} {'Contribution':>14s}")
print("-" * 60)
total_w = 0
total_wv = 0
for name, weight, value in components:
    if weight > 0 and value is not None:
        total_w += weight
        total_wv += weight * value
        print(f"{name:25s} {weight:8.2f} {value:8.4f} {weight * value:14.4f}")

quality = total_wv / total_w if total_w else 0
print(f"\nQuality = {total_wv:.4f} / {total_w:.2f} = {quality:.4f}")
print(f"Acceptance = {m.acceptance_rate:.4f}")
print(f"Cost = {m.total_cost:.4f}")
print(f"Score = {quality:.4f} * {m.acceptance_rate:.4f} / {m.total_cost:.4f} = {quality * m.acceptance_rate / max(m.total_cost, 1e-9):.6f}")
print(f"Library score: {opt_results[best_opt].score:.6f}")

> **Key Insight:** The score creates a three-way tension: (1) Better quality requires more complex circuits. (2) Stricter postselection improves quality but wastes shots. (3) More shots improve statistics but increase cost. The optimal configuration balances all three.

---
## 9. Factory Throughput: An Alternative Score

The factory throughput scorer optimizes for *yield* — accepted magic states per unit cost:

$$\text{throughput} = \frac{\text{acceptance} \times W}{\text{cost}}$$

In [ ]:
quiz(tracker, "q8_factory_vs_wac",
    question="Two scorers rank experiments differently. What determines which to use?",
    options=[
        "Always use WAC \u2014 it's the default",
        "Your operational goal: quality per state (WAC) vs production rate (factory throughput)",
        "Use factory throughput only on real hardware",
    ],
    correct=1, section="9. Factory throughput", bloom="evaluate",
    explanation="The choice of scorer encodes your priorities. WAC optimizes per-state quality. Factory throughput optimizes for a T-state production pipeline.")
checkpoint_summary(tracker, "9. Factory throughput")

In [ ]:
factory_config = ScoreConfig(
    name="factory_throughput",
    cheap_quality=QualityWeights(
        noisy_fidelity=0.3, logical_witness=0.4,
        codespace_rate=0.2, stability_score=0.1,
    ),
)

print(f"{'opt':>5s}  {'WAC Score':>12s}  {'Factory Score':>14s}  {'WAC Rank':>10s}  {'Fac Rank':>10s}")
wac_scores = {}
fac_scores = {}
for opt in [1, 2, 3]:
    m = opt_results[opt].metrics
    m.extra = {}
    s_wac = opt_results[opt].score
    s_ft, _, _ = factory_throughput_score(m, "cheap", factory_config)
    wac_scores[opt] = s_wac
    fac_scores[opt] = s_ft

wac_rank = sorted(wac_scores, key=wac_scores.get, reverse=True)
fac_rank = sorted(fac_scores, key=fac_scores.get, reverse=True)
for opt in [1, 2, 3]:
    print(f"{opt:>5d}  {wac_scores[opt]:>12.4f}  {fac_scores[opt]:>14.4f}  "
          f"{'#' + str(wac_rank.index(opt)+1):>10s}  {'#' + str(fac_rank.index(opt)+1):>10s}")

> **Observe:** The two scorers may rank configurations differently. Factory throughput penalizes circuit cost more heavily, favoring simpler circuits even if quality is slightly lower.

---
## Summary

| Concept | What you learned |
|---|---|
| **Noise model** | Each gate has a probability of error; readout is imperfect |
| **Transpilation** | Maps logical circuits to physical hardware; higher optimization can reduce gates |
| **Acceptance rate** | Fraction of shots surviving postselection; drops under noise |
| **Cost model** | Penalizes 2Q gates, depth, shots, runtime |
| **Failure modes** | Four categories classify the dominant weakness |
| **Score** | quality x acceptance / cost — the single optimization target |

> **Next:** Track A covers the physics in depth. Track C shows how the ratchet automates parameter search.

> **Dashboard Exercise:** Go to `00_dashboard.ipynb`. Try every combination of verification mode and optimization level. Can you find the configuration with the highest score?

---
## Final Assessment

In [ ]:
tracker.dashboard()
path = tracker.save()
print(f"\nProgress saved to: {path}")

---
## Navigation — Plan C

**→ Next: [Track C — Search](track_c_search.ipynb)**

*← Previous: [Track A — Physics](track_a_physics.ipynb) · [Dashboard](00_dashboard.ipynb) · [Start Here](../00_START_HERE.ipynb)*